# Results Analysis — German Credit (Credit-Risk Tabular Baseline)

**Dissertation:** Graph Neural Networks for Systemic Risk and Fraud Detection in Credit Systems  
**Track:** Credit risk (tabular baseline)  
**Student:** Chandan Nagavolu — M.Tech, BITS Pilani WILP

The German Credit dataset (~1000 applicants, good/bad risk) is the small,
well-understood baseline for the credit-risk track. Its job is to establish the
**credit-scorecard metrics** — KS, Gini and Weight-of-Evidence / Information
Value — on a tractable problem before scaling to larger relational datasets
(Home Credit, Lending Club). There is **no graph / GNN** here; the data is
purely tabular.

This notebook reads:
- the cross-model results table `reports/results_german_credit.csv`,
- the KS/Gini scorecard outputs under `reports/ks_gini/`, and
- the Information Value table `reports/german_credit_iv.csv`,

all produced by the run order documented in `data/README.md`.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110


def find_project_root(start: Path) -> Path:
    """Walk up from the notebook until the folder containing CLAUDE.md."""
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "CLAUDE.md").exists():
            return cand
    raise FileNotFoundError("Could not locate project root (CLAUDE.md).")


ROOT = find_project_root(Path.cwd())
RESULTS_CSV = ROOT / "reports" / "results_german_credit.csv"
KS_DIR = ROOT / "reports" / "ks_gini"
FIG_DIR = ROOT / "reports" / "figures"
IV_CSV = ROOT / "reports" / "german_credit_iv.csv"

MODEL_ORDER = ["logistic_regression", "random_forest", "xgboost"]
MODEL_PRETTY = {
    "logistic_regression": "Logistic Regression",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
}
METRICS = ["roc_auc", "pr_auc", "ks_statistic", "gini", "f1", "mcc"]
PRETTY = {
    "roc_auc": "ROC-AUC", "pr_auc": "PR-AUC", "ks_statistic": "KS",
    "gini": "Gini", "f1": "F1", "mcc": "MCC",
}
print("Project root:", ROOT)

## 1. Why these metrics?

German Credit is only mildly imbalanced (~30% bad), so standard metrics are
usable — but the **credit-industry discrimination metrics** are the headline
for this track because they are what a risk team would actually report on a
scorecard:

| Metric | Family | Why it matters here |
|---|---|---|
| **KS statistic** | Credit | Maximum separation between the cumulative "bad" and "good" score distributions. Rule of thumb: **KS > 0.40 = strong**. |
| **Gini** | Credit | `2*AUC - 1`; rank-ordering power on a 0–1 scale familiar to risk teams. |
| **PR-AUC** | Fraud/imbalance | Robust summary of performance on the minority (bad) class. |
| **MCC** | Balanced | Single balanced score over all four confusion-matrix cells. |
| **ROC-AUC / F1** | Standard | Reported for completeness / comparability with the other tracks. |

Because the dataset is tiny, a single 150-row test split gives a noisy KS/Gini,
so we read **both** the single held-out split and the **5-fold cross-validated**
mean ± std.

In [ ]:
raw = pd.read_csv(RESULTS_CSV, parse_dates=["timestamp"])
raw = raw.sort_values("timestamp").reset_index(drop=True)

# Keep the LATEST row per (model, split) so re-runs don't double-count.
latest = (
    raw.groupby(["model", "split"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)
latest[["model", "split", *METRICS]].round(4)

## 2. Model leaderboard — held-out test set

The three traditional baselines on the held-out test split. KS and Gini are the
discrimination numbers a credit team would quote.

In [ ]:
board = (
    latest[latest.split == "test"]
    .set_index("model")
    .reindex(MODEL_ORDER)
)
table = board[METRICS].rename(columns=PRETTY)
table.index = [MODEL_PRETTY[m] for m in table.index]
table.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(METRICS))
width = 0.25
for i, model in enumerate(MODEL_ORDER):
    vals = board.loc[model, METRICS].values.astype(float)
    ax.bar(x + (i - 1) * width, vals, width, label=MODEL_PRETTY[model], edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels([PRETTY[m] for m in METRICS])
ax.set_ylim(0, 1.05)
ax.axhline(0.40, color="red", ls="--", lw=1, label="KS/Gini = 0.40 (strong)")
ax.set_title("German Credit — baseline metrics (test split)")
ax.set_ylabel("Score")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 3. Cross-validated stability (5-fold)

The single-split numbers above sit on only ~150 test rows. The 5-fold CV
mean ± std (pipeline refit per fold) is the more trustworthy headline for a
dataset this small.

In [ ]:
cv_mean = latest[latest.split == "cv_mean"].set_index("model").reindex(MODEL_ORDER)
cv_std = latest[latest.split == "cv_std"].set_index("model").reindex(MODEL_ORDER)

if cv_mean[METRICS].notna().any().any():
    cv = pd.DataFrame(index=[MODEL_PRETTY[m] for m in MODEL_ORDER])
    for m in METRICS:
        cv[PRETTY[m]] = [
            f"{cv_mean.loc[k, m]:.3f} ± {cv_std.loc[k, m]:.3f}" for k in MODEL_ORDER
        ]
    display(cv)
else:
    print("No CV rows found — run the trainer with cross_validation.enabled: true.")

## 4. Credit scorecard — KS / Gini summary

From `src/evaluation/evaluate_german_credit_ks_gini.py`: per-model decile KS
tables, the continuous vs decile KS, and the band where separation peaks.

In [ ]:
summary = pd.read_csv(KS_DIR / "german_credit_ks_gini_summary.csv")
summary.round(4)

In [ ]:
# KS curves saved by the scorecard runner.
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(16, 5))
for ax, model in zip(np.atleast_1d(axes), MODEL_ORDER):
    img_path = FIG_DIR / f"ks_{model}_german_credit.png"
    if img_path.exists():
        ax.imshow(mpimg.imread(img_path))
    else:
        ax.text(0.5, 0.5, f"missing\n{img_path.name}", ha="center", va="center")
    ax.set_title(MODEL_PRETTY[model])
    ax.axis("off")
fig.tight_layout()
plt.show()

## 5. Feature screening — Weight-of-Evidence / Information Value

IV (computed on the train split only) ranks features by univariate predictive
power. Bands: <0.02 useless, 0.02–0.1 weak, 0.1–0.3 medium, 0.3–0.5 strong,
>0.5 suspicious (possible leakage).

In [ ]:
iv = pd.read_csv(IV_CSV)
display(iv.round(4))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(iv["feature"][::-1], iv["iv"][::-1], color="#4C72B0", edgecolor="white")
for band, x in [("weak", 0.02), ("medium", 0.1), ("strong", 0.3)]:
    ax.axvline(x, color="grey", ls="--", lw=1)
    ax.text(x, -0.6, band, rotation=90, fontsize=8, va="bottom", color="grey")
ax.set_xlabel("Information Value")
ax.set_title("German Credit — feature Information Value (train split)")
ax.grid(True, axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

## 6. Takeaways

- The traditional baselines (Logistic Regression, Random Forest, XGBoost)
  establish the **reference KS / Gini** for the credit-risk track. On this
  classic dataset, well-tuned models typically land around KS ≈ 0.30–0.45 /
  Gini ≈ 0.45–0.55 — read the CV mean ± std (§3) rather than the noisy single
  split for the headline.
- The **WoE / IV** screening (§5) identifies the strongest univariate
  discriminators (commonly `Checking account`, `Duration`, `Credit amount`),
  giving an interpretable, scorecard-style view alongside the model metrics.
- These numbers are the **baseline to beat** when the credit-risk track scales
  to the larger temporal / relational datasets and the TCN / autoencoder
  models in later chapters.